# Reinforcement Learning From AI Feedback (RLAIF) with Serverless customization on SageMaker AI

## Lab 2 - Reward Model

In this notebook, we are going to prepare the reward model for later on fine-tuning Qwen 2.5 - 7B Instruct

## Prerequisites

In [ ]:
import os
import boto3
from sagemaker.core.helper.session_helper import Session

PROFILE_NAME = "njourdan-Admin"

# Set AWS profile environment variable BEFORE importing sagemaker modules
os.environ['AWS_PROFILE'] = PROFILE_NAME
os.environ['AWS_DEFAULT_REGION'] = 'us-west-2'

boto_session = boto3.Session(profile_name=PROFILE_NAME, region_name='us-west-2')
sagemaker_session = Session(boto_session=boto_session)

role = boto_session.client("sts").get_caller_identity()["Arn"]

print(f"sagemaker role arn: {role}")
print(f"sagemaker bucket: {sagemaker_session.default_bucket()}")
print(f"sagemaker session region: {sagemaker_session.boto_region_name}")


sagemaker.config INFO - Not applying SDK defaults from location: /Library/Application Support/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /Users/njourdan/Library/Application Support/sagemaker/config.yaml
sagemaker role arn: arn:aws:sts::157476309474:assumed-role/Admin/njourdan-Isengard
sagemaker bucket: sagemaker-us-west-2-157476309474
sagemaker session region: us-west-2


## Prepare the reward prompt
### Inspect and upload to S3

TODO TEXT
We are going to use the [Human-Like-DPO-Dataset](https://huggingface.co/datasets/HumanLLMs/Human-Like-DPO-Dataset) that was created as part of research aimed at improving conversational fluency and engagement in large language models. It is suitable for formats like Direct Preference Optimization (DPO) to guide models toward generating more human-like responses, although in this example we'll use just the chosen response to implement RLAIF.

In [5]:
import json

with open("reward_prompt.json") as f:
    reward_prompt = json.load(f)
print(reward_prompt["prompt"])

You are an expert human evaluator assessing how human-like a text response is compared to a high-quality human-written reference. Given the original question, a ground truth response exemplifying ideal human tone, style, and language, and a new LLM-generated response, rate how closely the LLM response matches the ground thruth response in the following aspects:

- Naturalness and fluidity of language
- Appropriateness and consistency of tone
- Stylistic nuances and variability typical of human writing
- Coherence and logical flow without artificial phrasing or repetitive structures

Instructions:
- Read the question carefully.
- Read the ground truth response as the gold standard for tone, style, and language quality.
- Compare the model's response with the ground thruth response.
- Ignore spelling errors
- Don't over-index on completeness and length of the answer
- Assign a score from 0 to 1 representing how human-like the model's reply sounds:
   - 1 means indistinguishable from a na

In [ ]:
import boto3

s3_client = boto3.client('s3')
bucket_name = sagemaker_session.default_bucket()

file_name = 'reward_prompt.json'
prefix_key = "human-like-rlaif/{0}".format(file_name)
s3_client.upload_file(file_name, bucket_name, prefix_key)

reward_prompt_uri = "s3://{0}/{1}".format(bucket_name, prefix_key)
print(reward_prompt_uri)

╭─────────────────────────────── Traceback (most recent call last) ────────────────────────────────╮
│ /Users/njourdan/workspace/serverless-customization-workshop/.venv/lib/python3.13/site-packages/b │
│ oto3/s3/transfer.py:452 in upload_file                                                           │
│                                                                                                  │
│   449 │   │   │   filename, bucket, key, extra_args, subscribers                                 │
│   450 │   │   )                                                                                  │
│   451 │   │   try:                                                                               │
│ ❱ 452 │   │   │   future.result()                                                                │
│   453 │   │   # If a client error was raised, add the backwards compatibility layer              │
│   454 │   │   # that raises a S3UploadFailedError. These specific errors were only               │
│   455 │   │   # ever thrown for upload_parts but now can be thrown for any related               │
│                                                                                                  │
│ /Users/njourdan/workspace/serverless-customization-workshop/.venv/lib/python3.13/site-packages/s │
│ 3transfer/futures.py:111 in result                                                               │
│                                                                                                  │
│   108 │   │   │   # Usually the result() method blocks until the transfer is done,               │
│   109 │   │   │   # however if a KeyboardInterrupt is raised we want want to exit                │
│   110 │   │   │   # out of this and propagate the exception.                                     │
│ ❱ 111 │   │   │   return self._coordinator.result()                                              │
│   112 │   │   except KeyboardInterrupt as e:                                                     │
│   113 │   │   │   self.cancel()                                                                  │
│   114 │   │   │   raise e                                                                        │
│                                                                                                  │
│ /Users/njourdan/workspace/serverless-customization-workshop/.venv/lib/python3.13/site-packages/s │
│ 3transfer/futures.py:287 in result                                                               │
│                                                                                                  │
│   284 │   │   # Once done waiting, raise an exception if present or return the                   │
│   285 │   │   # final result.                                                                    │
│   286 │   │   if self._exception:                                                                │
│ ❱ 287 │   │   │   raise self._exception                                                          │
│   288 │   │   return self._result                                                                │
│   289 │                                                                                          │
│   290 │   def cancel(self, msg='', exc_type=CancelledError):                                     │
│                                                                                                  │
│ /Users/njourdan/workspace/serverless-customization-workshop/.venv/lib/python3.13/site-packages/s │
│ 3transfer/tasks.py:142 in __call__                                                               │
│                                                                                                  │
│   139 │   │   │   │   # task to the TransferFuture had failed) then execute the task's           │
│   140 │   │   │   │   # main() method.                                                           │
│   141 │   │   │   │   if not self._transfer_coordinator.don

### Register SageMaker AI Evaluator with the Reward Prompt

In [ ]:
from sagemaker.ai_registry.evaluator import Evaluator
from sagemaker.ai_registry.air_constants import REWARD_PROMPT

reward_prompt = Evaluator.create(
    name="humanlike-reward-prompt",
    type=REWARD_PROMPT,
    source=reward_prompt_uri,
    wait=True)
    
print(f"Reward prompt ARN: {reward_prompt.arn}")
reward_prompt_arn=reward_prompt.arn